# Pipeline Parallel Training Simulator Lab
__Designed by Edouard Oyallon__

This lab uses a **discrete time-step simulator** to study pipeline-parallel training schedules such as **GPipe**, **1F1B**, **DualPipeV**, and scripted variants. The simulator models **exact gradients**, i.e., behavior equivalent to standard training with **gradient accumulation** over a fixed global mini-batch.

## Modeling assumptions (pipeline-parallel training simulator)

### Pipeline structure
- The network is split into **D sequential stages** $S_0, S_1, \dots, S_{D-1}$.
- Each stage is mapped to a GPU via `stage_to_device`; there are **G GPUs** in total.
- One training iteration is divided into **N microbatches** that flow through the pipeline.

### Discrete-time model
- Time advances in integer ticks: $t = 0, 1, 2, \dots$.
- All operations have **integer durations**:
  - Forward: $T_f$
  - Backward-for-weights: $T_{bw}$
  - Backward-for-activations: $T_{ba}$
  - Optimizer update: $T_u$
  - Communication delay: $T_c$

### Per-stage compute operators
For each stage $i$ and microbatch $m$, the simulator uses four compute ops:

- **$F_i^m$**: Forward on microbatch $m$ at stage $i$ (duration $T_f$)  
  Stores two activation “handles” (a proxy for memory residency):
  - `act_w` (needed for weight-grad backward)
  - `act_a` (needed for activation backward)

- **$A_i^m$**: Backward-for-activations on microbatch $m$ at stage $i$ (duration $T_{ba}$)  
  Frees `act_a` and triggers **sending** $dA$ to stage $i-1$ (unless $i=0$).

- **$W_i^m$**: Backward-for-weights on microbatch $m$ at stage $i$ (duration $T_{bw}$)  
  Frees `act_w` and can only be done if $A_i^m$ has been obtained. (This corresponds to computing/accumulating $dW$.)

- **$U_i$**: Optimizer/update for stage $i$ (duration $T_u$)  
  Executed **once per iteration**, after all microbatches have completed the required backward work on that stage.

### Per-GPU execution model (compute serialization + comm overlap)
Each GPU is modeled with two logical resources:

**Compute resource**
- Stages placed on the **same GPU cannot compute in parallel** (compute is serialized per GPU).
- At most one compute op runs per GPU at a time, even if it belongs to different stages.

**Communication resource**
- Communication is represented explicitly as **SEND events**.
- SENDs may **overlap** with compute.
- Optionally, SENDs can be serialized per GPU (one SEND at a time) via `serialize_send_per_gpu`.

### Communication model (explicit SEND, free RECV)
The simulator models point-to-point communication between consecutive stages:

- **Forward SEND:** after $F_i^m$, stage $i$ sends activation $A_i^m$ to stage $i+1$  
  (only if $i < D-1$).

- **Backward SEND:** after $A_i^m$, stage $i$ sends activation gradient $dA_i^m$ to stage $i-1$  
  (only if $i > 0$).

**RECV is free/automatic:** arrivals are delivered when their delay elapses; no explicit RECV op is scheduled.

Local transfers:
- If two consecutive stages are on the **same GPU**, communication is treated as **instantaneous** (effectively $T_c=0$ for that edge).
- Otherwise, SEND incurs a transit delay of $T_c$.

### Memory proxy (activation locking)
We track transient training memory via activation “liveness”:
- `act_w` is allocated by $F$ and freed by $W$.
- `act_a` is allocated by $F$ and freed by $A$.

Per-tick “GPU memory” is reported as a **proxy count** of live activations on that GPU:

$$
\mathrm{mem}(g,t) = \#\text{live act\_w on } g \;+\; \#\text{live act\_a on } g.
$$

### Dependencies and legality
An operation can start only if:
- Its required inputs have arrived (via automatic RECV),
- Ativations gradients must be computed before weights gradients.
- Its required activations are still live (`act_w` for $W$, `act_a` for $A$),
- The stage and its GPU compute resource are free at time $t$,
- For non-local edges, the corresponding channel has capacity to accept a SEND (backpressure). Not used in this lab

These assumptions are sufficient to reproduce and compare pipeline schedules while making compute/comm overlap, GPU co-location, and activation lifetimes explicit.

## Pipeline conditions

For a fixed stage, we denote by $O_m$ (or $Om$) the operation $O \in \{F, A, W\}$ applied to the $m$-th microbatch.

### Do we really need to track microbatch IDs inside a stage?

**Question 1 — Show that, for any stage and any valid instruction trace, we can assume (without loss of generality) that microbatches are processed in increasing order.**  
More precisely, prove that for any two operations of the same type $O$ executed on the same stage, if they appear as $O_{m_2}$ followed by $O_{m_1}$ with $m_1 < m_2$, then there exists an equivalent valid trace where their order is swapped. Conclude that, for a given stage, it is sufficient to write only the sequence of operator types (e.g., $F,F,A,W,\dots$) without specifying the microbatch id.

---


**Question 2 — Remind why there is a specific order between activation- and weight-gradient computations.**
More precisely, explain why (for a fixed stage and microbatch $m$) the simulator enforces the dependency $A_m \rightarrow W_m$.

---


## Simulator code (reference implementation)

Below is the reference implementation of our **pipeline-parallel training simulator**. 

In [ ]:
"""
Pipeline-parallel training simulator (discrete time-steps) with split backward + GPU grouping + comm "send" events.

Per stage + microbatch compute ops:
- Fm : forward for microbatch m
- Wm : backward-for-weights (dW) for microbatch m  -> frees act_w
- Am : backward-for-activations (dA) for microbatch m -> frees act_a, and triggers sending dA upstream
- U  : optimizer update (once per iteration, after all microbatches' A are done)

Key features requested:
1) Each stage i is assigned to a GPU (stage_to_device). Stages on the same GPU cannot compute in parallel.
2) Communication shows up explicitly as SEND events:
   - Forward: send activation A_i^m from stage i to stage i+1 after Fm
   - Backward: send activation gradient dA_i^m from stage i to stage i-1 after Am
3) We assume RECV is "overlappable"/free: arrivals happen automatically when their delay elapses.
   SEND consumes a separate per-GPU comm engine (optional serialization); compute and send can overlap.
4) Final summary shows each stage row prefixed by its GPU id.

Notes:
- This is a *time-tick* simulator (t = 0..T-1). Ops have integer durations (Tf, Tbw, Tba, Tu, Tc).
- For simplicity, activations carry only the microbatch id. You can extend payloads with sizes later.
"""


from __future__ import annotations
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Callable, Deque
from collections import deque, defaultdict
from functools import partial


# -----------------------------
# Pretty printing helpers
# -----------------------------
def cell(x: str, w: int = 3) -> str:
    """Fixed-width cell for timeline."""
    return f"{x:>{w}}"

def stage_label(gpu: int, stage: int) -> str:
    return f"GPU{gpu}:S{stage}"


# -----------------------------
# Channels: delayed delivery + finite capacity
# -----------------------------
@dataclass
class Channel:
    """Finite-capacity FIFO with transit delay Tc. Enqueue at send-time; becomes recv-able at (t + Tc)."""
    capacity: int
    delay: int
    _q: Deque[Tuple[int, int]] = field(default_factory=deque)  # (available_time, microbatch)

    def can_send(self) -> bool:
        return len(self._q) < self.capacity

    def send(self, microbatch: int, t: int) -> None:
        if not self.can_send():
            raise RuntimeError("Channel full (backpressure).")
        self._q.append((t + self.delay, microbatch))
    
    def send_instantaneous(self, microbatch: int, t: int) -> None:
        if not self.can_send():
            raise RuntimeError("Channel full (backpressure).")
        self._q.append((t, microbatch))

    def pop_ready(self, t: int) -> List[int]:
        """Pop and return all microbatches that have arrived by time t."""
        out = []
        while self._q and self._q[0][0] <= t:
            _, m = self._q.popleft()
            out.append(m)
        return out


# -----------------------------
# Config + state
# -----------------------------
@dataclass
class PipelineConfig:
    D: int
    N: int
    # homogeneous compute
    Tf: int = 2
    Tbw: int = 1
    Tba: int = 1
    Tu: int = 1

    # communication
    Tc: int = 1
    edge_capacity: int = 999

    # GPU mapping
    stage_to_device: Optional[List[int]] = None

    # comm engine model:
    # If True: one SEND at a time per GPU (but still overlaps compute).
    # If False: sends are "free" as well (still appear in logs, but no serialization).
    # In this lab, this won't be that useful as PP are supposed to send ASAP their results.
    serialize_send_per_gpu: bool = True



@dataclass
class StageState:
    stage_id: int

    # Separate activation stores
    act_w_store: Dict[int, bool] = field(default_factory=dict)  # m -> stored?
    act_a_store: Dict[int, bool] = field(default_factory=dict)  # m -> stored?

    # Progress per microbatch
    bw_done: Dict[int, bool] = field(default_factory=dict)
    ba_done: Dict[int, bool] = field(default_factory=dict)

    # Counters
    f_done: int = 0
    w_done: int = 0
    a_done: int = 0
    updated: bool = False


# policy returns desired compute op per stage: ("F"/"W"/"A"/"U"/None, mb or None)
Policy = Callable[["PipelineSim", int], List[Tuple[Optional[str], Optional[int]]]]


# -----------------------------
# Simulator
# -----------------------------
class PipelineSim:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.t = 0

        # GPU mapping
        if cfg.stage_to_device is None:
            self.stage_to_device = list(range(cfg.D))
        else:
            if len(cfg.stage_to_device) != cfg.D:
                raise ValueError("stage_to_device must have length D.")
            self.stage_to_device = cfg.stage_to_device

        # device->stages
        self.device_to_stages: Dict[int, List[int]] = defaultdict(list)
        for s, dev in enumerate(self.stage_to_device):
            self.device_to_stages[dev].append(s)
        for dev in self.device_to_stages:
            self.device_to_stages[dev].sort()

        # stage states
        self.stages: List[StageState] = [StageState(i) for i in range(cfg.D)]

        self.fwd_ch: Dict[int, Channel] = {
        i: Channel(cfg.edge_capacity, cfg.Tc)
        for i in range(cfg.D - 1)
        }

        self.bwd_ch: Dict[int, Channel] = {
        i: Channel(cfg.edge_capacity, cfg.Tc)
        for i in range(1, cfg.D)
        }

        # readiness queues (microbatch ids)
        self.fwd_ready: List[Deque[int]] = [deque() for _ in range(cfg.D)]
        self.bw_ready: List[Deque[int]] = [deque() for _ in range(cfg.D)]
        self.ba_ready: List[Deque[int]] = [deque() for _ in range(cfg.D)]

        # initialize inputs at stage 0
        for m in range(cfg.N):
            self.fwd_ready[0].append(m)

        # compute occupancy: per-stage and per-GPU
        self.stage_busy_until: List[int] = [0] * cfg.D
        self.gpu_compute_busy_until: Dict[int, int] = defaultdict(int)

        # send occupancy: per-GPU comm engine (optional)
        self.gpu_send_busy_until: Dict[int, int] = defaultdict(int)
        # persistent pending sends per stage: items are ("fwd"/"bwd", microbatch)
        self.pending_send_q: List[Deque[Tuple[str,int]]] = [deque() for _ in range(cfg.D)]


        # timeline logs
        # compute_timeline[t][i] = label like "F3", "W2", "A0", "U", "."
        # send_timeline[t][i] = label like "→1" or "←3" or "."
        self.compute_timeline: List[List[str]] = []
        self.send_timeline: List[List[str]] = []
        # memory timeline per tick: list of dicts gpu->counts
        self.gpu_mem_timeline: List[Dict[int, Dict[str, int]]] = []



    # ---------- helpers: resources ----------
    def _gpu(self, stage: int) -> int:
        return self.stage_to_device[stage]

    def stage_compute_free(self, i: int, t: int) -> bool:
        return t >= self.stage_busy_until[i] and t >= self.gpu_compute_busy_until[self._gpu(i)]

    def gpu_send_free(self, gpu: int, t: int) -> bool:
        if not True:
            return True
        return t >= self.gpu_send_busy_until[gpu]

    def occupy_compute(self, i: int, t: int, dur: int) -> None:
        self.stage_busy_until[i] = t + dur
        gpu = self._gpu(i)
        self.gpu_compute_busy_until[gpu] = t + dur

    def occupy_send(self, gpu: int, t: int, dur: int) -> None:
        if not True:
            return
        self.gpu_send_busy_until[gpu] = t + dur

    # ---------- comm progression (RECV is free/automatic) ----------
    def progress_receives(self, t: int) -> None:
        # forward receives enable forward at next stage
        for i in range(self.cfg.D - 1):
            arrived = self.fwd_ch[i].pop_ready(t)
            for m in arrived:
                self.fwd_ready[i + 1].append(m)

        # backward receives enable BW at previous stage
        for i in range(1, self.cfg.D):
            arrived = self.bwd_ch[i].pop_ready(t)  # channel i corresponds to (i+1)->i
            for m in arrived:
                self.ba_ready[i - 1].append(m)

    # ---------- can-start compute ----------
    def can_F(self, i: int, t: int) -> bool:
        if not self.fwd_ready[i]:
            return False
        # need space to enqueue activation to next stage unless last stage
        if i < self.cfg.D - 1 and not self.fwd_ch[i].can_send():
            return False
        return self.stage_compute_free(i, t)

    def can_W(self, i: int, t: int) -> bool:
        if not self.bw_ready[i]:
            return False
        m = self.bw_ready[i][0]
        s = self.stages[i]
        return s.act_w_store.get(m, False) and self.stage_compute_free(i, t)

    def can_A(self, i: int, t: int) -> bool:
        if not self.ba_ready[i]:
            return False
        m = self.ba_ready[i][0]
        s = self.stages[i]
        # need space to send dA unless stage 0
        if i > 0 and not self.bwd_ch[i].can_send():
            return False
        return s.act_a_store.get(m, False) and self.stage_compute_free(i, t)

    def can_U(self, i: int, t: int) -> bool:
        s = self.stages[i]
        if s.updated:
            return False
        # exact gradients + grad accumulation => update after all A done on that stage
        if s.w_done < self.cfg.N:
            return False
        return self.stage_compute_free(i, t)
    
    def is_local_fwd(self, i: int) -> bool:
        return (i < self.cfg.D - 1 and self._gpu(i) == self._gpu(i + 1)) or self.cfg.Tc==0

    def is_local_bwd(self, i: int) -> bool:
        return (i > 0 and self._gpu(i) == self._gpu(i - 1)) or self.cfg.Tc==0


    # ---------- send actions (consume optional comm engine, overlap compute) ----------
    def try_send(self, src_stage: int, direction: str, microbatch: int, t: int) -> bool:
        """
        Attempt to perform a SEND at time t:
        direction = "fwd" (to i+1) or "bwd" (to i-1)
        Returns True if send happened.
        """
        gpu = self._gpu(src_stage)
        if not self.gpu_send_free(gpu, t):
            return False

        if direction == "fwd":
            if src_stage >= self.cfg.D - 1:
                return False
            if not self.fwd_ch[src_stage].can_send(): # EO: +1?
                return False
            self.fwd_ch[src_stage].send(microbatch, t) # EO: +1?
            self.occupy_send(gpu, t, self.cfg.Tc)
            return True

        if direction == "bwd":
            if src_stage <= 0:
                return False
            if not self.bwd_ch[src_stage].can_send():
                return False
            self.bwd_ch[src_stage].send(microbatch, t)
            self.occupy_send(gpu, t, self.cfg.Tc)
            return True

        raise ValueError("direction must be 'fwd' or 'bwd'")

    def progress_sends(self, t: int, send_row: List[str]) -> None:
        # build one "head-of-queue" candidate per stage
        candidates = []
        for s in range(self.cfg.D):
            if self.pending_send_q[s]:
                direction, mb = self.pending_send_q[s][0]
                prio = 0 if direction == "bwd" else 1  # bwd first
                candidates.append((prio, s, direction, mb))

        candidates.sort()

        for _, s, direction, mb in candidates:
            ok = self.try_send(s, direction, mb, t)
            if ok:
                self.pending_send_q[s].popleft()
                send_row[s] = (f"→{mb}" if direction == "fwd" else f"←{mb}")
            else:
                # still pending; keep it queued and mark blocked for visibility
                send_row[s] = "S!"


    # ---------- compute ops (may enqueue a pending send) ----------
    def do_F(self, i: int, t: int) -> Tuple[str, Optional[Tuple[str, int]]]:
        m = self.fwd_ready[i].popleft()
        self.occupy_compute(i, t, self.cfg.Tf)

        st = self.stages[i]
        st.f_done += 1
        # forward stores both activation types
        st.act_w_store[m] = True
        st.act_a_store[m] = True
        st.bw_done[m] = False
        st.ba_done[m] = False


        if i == self.cfg.D - 1:
            # last stage generates initial upstream grad => BA ready
            self.ba_ready[i].append(m)
            return f"F{m}", None

        if self.is_local_fwd(i):
            self.fwd_ch[i].send_instantaneous(m, t)            
            return f"F{m}", None
        else:
            # schedule a forward send of activations
            return f"F{m}", ("fwd", m)

    def do_W(self, i: int, t: int) -> Tuple[str, Optional[Tuple[str, int]]]:
        m = self.bw_ready[i].popleft()
        self.occupy_compute(i, t, self.cfg.Tbw)

        st = self.stages[i]
        st.bw_done[m] = True
        st.w_done += 1
        # free act_w
        st.act_w_store[m] = False

        return f"W{m}", None

    def do_A(self, i: int, t: int) -> Tuple[str, Optional[Tuple[str, int]]]:
        m = self.ba_ready[i].popleft()
        self.occupy_compute(i, t, self.cfg.Tba)

        st = self.stages[i]
        st.ba_done[m] = True
        st.a_done += 1
        # free act_a
        st.act_a_store[m] = False

        # after A, W becomes ready
        self.bw_ready[i].append(m)

        if i == 0:
            return f"A{m}", None

        if self.is_local_bwd(i):
            self.bwd_ch[i].send_instantaneous(m, t)     
            return f"A{m}", None
        else:
            # schedule a backward send of dA
            return f"A{m}", ("bwd", m)
    

    def do_U(self, i: int, t: int) -> Tuple[str, Optional[Tuple[str, int]]]:
        self.occupy_compute(i, t, self.cfg.Tu)
        self.stages[i].updated = True
        return "U", None

    # ---------- termination ----------
    def done(self) -> bool:
        return all(s.updated for s in self.stages)
    

    # ---------- step ----------
    def step(self, t: int, policy: Policy) -> None:
        self.t = t

        # 1) Deliver any receives whose delay elapsed (free/automatic)
        self.progress_receives(t)

        # 2) Decide compute ops
        desired_comp = policy(self, t)

        compute_row = ["." for _ in range(self.cfg.D)]
        send_row = ["." for _ in range(self.cfg.D)]

        
        self.progress_sends(t, send_row)

        # 3) Execute compute ops (subject to GPU compute serialization)
        for i in range(self.cfg.D):
            op = desired_comp[i]

            if op == "A" and self.can_A(i, t):
                lbl, pending = self.do_A(i, t)
                compute_row[i] = lbl
                if pending:
                    direction, mb = pending
                    self.pending_send_q[i].append((direction, mb))

            elif op == "W" and self.can_W(i, t):
                lbl, pending = self.do_W(i, t)
                compute_row[i] = lbl

            elif op == "F" and self.can_F(i, t):
                lbl, pending = self.do_F(i, t)
                compute_row[i] = lbl
                if pending:
                    direction, mb = pending
                    self.pending_send_q[i].append((direction, mb))

            elif op == "U" and self.can_U(i, t):
                lbl, pending = self.do_U(i, t)
                compute_row[i] = lbl

            else:
                compute_row[i] = "."


        self.compute_timeline.append(compute_row)
        self.send_timeline.append(send_row)
        #self.gpu_mem_timeline.append(self.memory_snapshot())


    # ---------- rendering ----------
    def print_tick(self, t: int) -> None:
        comp = " ".join(f"{cell(self.compute_timeline[t][i])}" for i in range(self.cfg.D))
        snd = " ".join(f"{cell(self.send_timeline[t][i])}" for i in range(self.cfg.D))
        header = " ".join(f"{cell('S'+str(i))}" for i in range(self.cfg.D))
        print(f"t={t:>3}  {header}")
        print(f"      C  {comp}")
        print(f"      S  {snd}\n")

    def print_summary(self, show_time_header: bool = True) -> None:
        """
        Final print: rows are stages, prefixed with the GPU hosting the stage.
        Shows both compute timeline and send timeline per stage.
        """

        # GPU blocks (optional) to visually confirm co-location
        print("\n=== GPU placement ===")
        for dev in sorted(self.device_to_stages.keys()):
            stages = ", ".join(str(s) for s in self.device_to_stages[dev])
            print(f"GPU {dev}: stages [{stages}]")

        T = len(self.compute_timeline)
        if T == 0:
            print("No timeline to display.")
            return

        # left column width
        left_w = max(len(stage_label(self._gpu(i), i)) for i in range(self.cfg.D)) + 2

        def time_header_line() -> str:
            return " " * left_w + "".join(cell(str(t % 10)) for t in range(T))

        print("\n=== PIPELINE TIMELINE (Compute and Send) ===")
        print("Legend: Fm/Wm/Am/U for compute; sends: →j:m (to next), ←j:m (to prev), S! blocked, . idle\n")
        if show_time_header:
            print("  "+time_header_line())

        for i in range(self.cfg.D):
            gpu = self._gpu(i)
            prefix = stage_label(gpu, i).ljust(left_w)

            comp_line = "".join(cell(self.compute_timeline[t][i]) for t in range(T))
            if self.cfg.Tc > 0:
                send_line = "".join(cell(self.send_timeline[t][i]) for t in range(T))

            print(f"{prefix}C {comp_line}")
            if self.cfg.Tc > 0:
                print(f"{'':<{left_w}}S {send_line}")

        print("\nNote: stages on the same GPU are compute-serialized (one compute op at a time per GPU).")



# -----------------------------
# Example policies
# -----------------------------
def greedy_policy(sim: PipelineSim, t: int) -> Dict[int, Tuple[Optional[str], Optional[int]]]:
    """
    Greedy compute policy (per stage), returned as a dict:
      prefer F, then A, then W, then U, else idle.
    Keys are stage indices.
    """
    out_comp: Dict[int, Tuple[Optional[str], Optional[int]]] = {}

    for i in reversed(range(sim.cfg.D)):
        if sim.can_F(i, t):
            out_comp[i] = "F"
        elif sim.can_A(i, t):
            out_comp[i] = "A"
        elif sim.can_W(i, t):
            out_comp[i] = "W"
        elif sim.can_U(i, t):
            out_comp[i] = "U"
        else:
            out_comp[i] = None

    return out_comp

def policy_from_script(sim: PipelineSim, t: int, script_gpu=None) -> Dict[int, Tuple[Optional[str], Optional[int]]]:
    """
    Script-driven compute policy with per-GPU queues.

    This policy lets you prescribe an explicit sequence of compute operations for each GPU
    using a compact token script (e.g., "0F 1F 1A 1W 0U"). Unlike a greedy policy, it does
    not search for "best" work: it follows the script order strictly, only advancing when
    the next scripted op is legal and the GPU is available.
    """

    if not hasattr(policy_from_script, "_queues"):
        policy_from_script._queues = {}  # (id(sim), gpu) -> deque[(stage,op)]

    out_comp: Dict[int, Tuple[Optional[str], Optional[int]]] = {i:None for i in range(sim.cfg.D)}

    def is_legal(stage: int, op: str) -> bool:
        if op == "F":
            return sim.can_F(stage, t)
        if op == "W":
            return sim.can_W(stage, t)
        if op == "A":
            return sim.can_A(stage, t)
        if op == "U":
            return sim.can_U(stage, t)
        return False

    scripted_gpus = set(script_gpu.keys()) if isinstance(script_gpu, dict) else set()
    if not scripted_gpus:
        return out_comp

    sid = id(sim)

    for gpu in scripted_gpus:
        gpu = int(gpu)
        key = (sid, gpu)

        # init queue once
        if key not in policy_from_script._queues:
            toks = re.findall(r"(\d+)\s*([FWAU])", str(script_gpu[gpu]))
            policy_from_script._queues[key] = deque((int(s), op) for s, op in toks)

        q = policy_from_script._queues[key]
        if not q:
            continue

        # GPU compute must be free to start anything
        if t < sim.gpu_compute_busy_until[gpu]:
            continue

        stage, op = q[0]

        # IMPORTANT: do NOT drop. If the token targets a stage not on this GPU, just wait.
        if sim._gpu(stage) != gpu:
            continue

        # Pop only if legal now; otherwise wait.
        if is_legal(stage, op):
            out_comp[stage] = op
            q.popleft()

    return out_comp


# -----------------------------
# Demo
# -----------------------------
if __name__ == "__main__":
    cfg = PipelineConfig(
        D=4,
        N=6,
        Tf=1, Tbw=1, Tba=1, Tu=1,
        Tc=0,               # make comm visible
        edge_capacity=999,
        stage_to_device=[0, 1, 2, 3],
    )

    sim = PipelineSim(cfg)
    policy = greedy_policy
    

    T = 200
    for t in range(T):
        sim.step(t, policy)
        # sim.print_tick(t)  # uncomment for step-by-step view
        if sim.done():
            break
        

    sim.print_summary(show_time_header=True)
    sim.print_memory_timeline()


**Question 3 — What are `is_local_fwd` and `is_local_bwd`? What do they simulate?**

Study the main simulator parameters (`D`, `N`, `Tf`, `Tbw`, `Tba`, `Tu`, `Tc`, `stage_to_device`, `serialize_send_per_gpu`) and run a few experiments by changing them.

**Question 4 — What does `greedy_policy` simulate?**
In particular, describe its priority rule (which ops it prefers and in what order) and what kind of pipeline behavior/schedule it tends to produce in practice. (see https://arxiv.org/abs/1811.06965)

Study the scripted policy interface and how it is plugged into the simulator. The scripted policy lets you prescribe, **for each GPU**, an explicit sequence of actions written as tokens of the form `stageIdxAction` (e.g., `0F`, `3A`, `2W`, `1U`). At each tick, the policy attempts to execute the **next token** for each GPU, in order, whenever it is legal.

**Question 5 — Is `policy = partial(policy_from_script, script_gpu=script_gpu)` correct? If not, fix it. Also give a condition for when a policy is “correct”.**

In [ ]:
cfg = PipelineConfig(
        D=4,
        N=6,
        Tf=1, Tbw=1, Tba=1, Tu=1,
        Tc=0,               # make comm visible
        edge_capacity=999,
        stage_to_device=[0, 0, 1, 1],
    )

sim = PipelineSim(cfg)

script_gpu = {
0: "0F 1F 0F 1F 1A 1W 0W 0A 0U 1U",
1: "2F 2F 3F 3F 3A 3A 3W 2A 2A 2W 2W 2U 3U",
}


policy = partial(policy_from_script, script_gpu=script_gpu)
# To clean potentially the policy
if hasattr(policy_from_script, "_queues"):
    policy_from_script._queues.clear()

T = 200
for t in range(T):
    sim.step(t, policy)
    # sim.print_tick(t)  # uncomment for step-by-step view
    if sim.done():
        break
    

sim.print_summary(show_time_header=True)
sim.print_memory_timeline()

Study how the simulator tracks activation liveness (`act_a_store`, `act_w_store`) and how this can be used as a simple proxy for GPU memory usage.

**Question 6 — Implement a per-tick “memory estimator” and a visualization of memory over time.**  
More precisely:
1. Write a function `memory_snapshot()` that, at a given tick, returns **per-GPU** counts of live buffers:
   - `act_a`: number of microbatches whose activation-backward buffers are still stored
   - `act_w`: number of microbatches whose weight-backward buffers are still stored
   - `total = act_a + act_w`
   (Each live buffer counts as 1 unit; this is a proxy for memory.)

2. Write a function `print_memory_timeline(mem_kind=...)` that prints a **time series** of this proxy across the whole simulation, for `mem_kind ∈ {"act_a","act_w","total"}`.

These two functions could be used to study how different schedules (policies, $T_c$, stage-to-GPU placement, etc.) affect peak memory.

_Hint:_ Check how `print_summary` is implemented.


Study the classic **1F1B** pipeline schedule in the special case where **each stage is placed on its own GPU** (i.e., `stage_to_device = [0,1,2,...,D-1]`). We want to drive the simulator using `policy_from_script`, which expects a dictionary `script_gpu` mapping each GPU to a whitespace-separated list of tokens of the form `stageIdxAction` (e.g., `2F`, `2A`, `2W`, `2U`). (see fig-8 https://arxiv.org/pdf/1806.03377)

**Question 7 — Write a function that automatically generates a closed-form 1F1B script for `policy_from_script` with warmup, then run the simulator with it.**  
More precisely:

1. Implement `make_1f1b_script_gpu(D, N) -> Dict[int, str]` that returns a `script_gpu` dictionary such that:
   - GPU id equals stage id (one stage per GPU).
   - For each stage $s$, the script follows the **1F1B structure**:
     - **Warmup:** run $(D-s)$ forwards (capped by $N$) to fill the pipeline.
     - **Steady state:** alternate **backward then forward**, i.e., repeat `(A, W, F)` until all remaining forwards are issued.
     - **Cooldown:** finish the remaining backwards, i.e., repeat `(A, W)` until all microbatches have completed backward.
     - **Update:** end with `U`.

2. Use your generator to build `script_gpu`, plug it into:
   - `policy = partial(policy_from_script, script_gpu=script_gpu)`,
   and simulate until `sim.done()`.

3. Print both:
   - the pipeline timeline (`sim.print_summary(...)`), and
   - the per-GPU memory proxy over time (`sim.print_memory_timeline(...)`).

Compare it to `GPipe`.


In [ ]:
from typing import Dict, List

def make_1f1b_script_gpu(D: int, N: int) -> Dict[int, str]:
    """
    Closed-form 1F1B script generator for your policy_from_script format.

    Assumptions (matching your description):
      - GPU id == stage id (1 stage per GPU)
      - Warmup forwards on stage s: (D - s) forwards (capped by M)
      - Then alternate: backward (A then W) + forward (F) until all forwards are done
      - Then cooldown: remaining backwards (A then W)
      - Then update U

    Returns:
      script_gpu: { stage: "0F 0F 0F 0F 0A 0W 0F ... 0U", ... }
    """
    if D <= 0:
        raise ValueError("D must be > 0")
    if N < 0:
        raise ValueError("M must be >= 0")

    script_gpu: Dict[int, str] = {}

    for s in range(D):
        pass
        # Fill in

    return script_gpu
D=4
N=8
cfg = PipelineConfig(
        D=D,
        N=N,
        Tf=1, Tbw=1, Tba=1, Tu=1,
        Tc=0,               # make comm visible
        stage_to_device=[0, 1, 2, 3],
    )

sim = PipelineSim(cfg)
script_gpu = make_1f1b_script_gpu(D, N)
policy = partial(policy_from_script, script_gpu=script_gpu)

T = 200
for t in range(T):
    sim.step(t, policy)
    # sim.print_tick(t)  # uncomment for step-by-step view
    if sim.done():
        break
    

sim.print_summary(show_time_header=True)
sim.print_memory_timeline()

Study alternative pipeline schedules by designing your own policy function and comparing its timeline/memory to 1F1B.

**Question 8 — Implement a custom schedule policy `ZBH2_policy` with an explicit warmup phase, then simulate it.**  
More precisely: (see https://arxiv.org/pdf/2401.10241)

1. Write a policy function `ZBH2_policy(sim, t)` that returns, for each stage $i$, the next operation to attempt at time $t$.
   - During a **warmup window**, force stage $i$ to run forward:
     - if $t < D - 2i - 1$, schedule `F` on stage $i$
   - After warmup, choose operations greedily with the priority:
     - `A` (if legal) → `F` → `W` → `U` → idle.

2. Run the simulator with:
   - $D=4$, $N=8$
   - one stage per GPU: `stage_to_device=[0,1,2,3]`
   - no communication cost: $T_c = 0$ (local/instant transfers)

3. Print your results.

Finally, briefly comment on how the warmup rule changes the early-stage behavior compared to a purely greedy policy, and what effect it has on bubbles and peak activation memory. Is $T_c=0$ important?


In [ ]:
def ZBH2_policy(sim: PipelineSim, t: int) -> Dict[int, Tuple[Optional[str], Optional[int]]]:
    out_comp: Dict[int, Tuple[Optional[str], Optional[int]]] = {}
    
    for i in range(sim.cfg.D):
        pass
        # Fill in here

    return out_comp

D=4
N=8
T_comm = 0
cfg = PipelineConfig(
        D=D,
        N=N,
        Tf=1, Tbw=1, Tba=1, Tu=1,
        Tc=0,               # make comm visible
        edge_capacity=999,
        stage_to_device=[0, 1, 2, 3],
    )

sim = PipelineSim(cfg)
policy = ZBH2_policy


T = 200
for t in range(T):
    sim.step(t, policy)
    # sim.print_tick(t)  # uncomment for step-by-step view
    if sim.done():
        break
    

sim.print_summary(show_time_header=True)
sim.print_memory_timeline()

Study the **DualPipeV** schedule (two stages per GPU) using the scripted-policy interface. See for instance https://github.com/deepseek-ai/DualPipe/tree/main?tab=readme-ov-file and https://huggingface.co/blog/NormalUhr/deepseek-dualpipe#36-overall-scheduling-the-step-methods-8-stages.

**Question 9 — Implement `build_dualpipev_script_gpu(D, N, stage_to_device)` (DualPipeV script generator).**  
Assume:
- $D$ is even and $N \ge D$.
- There are $R = D/2$ ranks (GPUs), and each rank hosts exactly two stages.
- For rank `r`, recover its two local stages with `s0, s1` being the lower and higher stages of a given rank.

Write a function that returns `script_gpu: Dict[int, str]`, compatible with `policy_from_script`, by emitting (per rank) the DualPipeV token pattern built from the 8 phase counts:

For each rank `r`, append tokens in this order:
- Phase 1: repeat `s0F` `(R - r - 1) * 2` times
- Phase 2: repeat `s0F s1F` `r + 1` times
- Phase 3: repeat `s1A s1W s1F` `R - r - 1` times
- Phase 4: repeat `s0F s1A s1W s1F s0A s0W` `N - 2 * R + r + 1` times
- Phase 5: repeat `s1A s1W s1F s0A s0W` `R - r - 1` times
- Phase 6: perform `r + 1` iterations of `s1A (s1W or delayed); s0A (s0W or delayed)`, where after the halfway point you start **delaying** some `W` operations into a FIFO queue to stack them
- Phase 7: repeat `s0W s0A` `R - r - 1` times
- Phase 8: flush the delayed-`W` queue for `r+1` pops (emit `stageW` for each pop)
- Finish with `s0U s1U`


In [ ]:
from typing import Dict, List, Tuple, Optional
from collections import defaultdict, deque


def build_dualpipev_script_gpu(
    D: int,
    N: int,
    stage_to_device: Optional[List[int]],
) -> Dict[int, str]:
    """
    DualPipeV-style script generator in policy_from_script format.
    Output: {rank/gpu: "stageOp stageOp ..."} with ops in {F,A,W,U}.

    Assumptions:
      - D even, 2 stages per rank (R = D//2).
      - N >= D (typical requirement).
      - If stage_to_device is given: each GPU hosts exactly 2 stages.
      - Else: use "v mapping" rank r owns stages (r, D-1-r).
    """
    if D % 2 != 0:
        raise ValueError("DualPipeV requires even D (2 stages per rank).")
    if N < D:
        raise ValueError("DualPipeV typically requires N >= D.")

    R = D // 2

    # Collect tokens per rank as (stage, op) pairs
    script: Dict[int, List[Tuple[int, str]]] = {r: [] for r in range(R)}

    for r in range(R):
        pass
        # FIll in

    # Convert to strings consumable by policy_from_script regex: r"(\d+)\s*([FWAU])"
    return {
        r: " ".join(f"{stage}{op}" for stage, op in toks)
        for r, toks in script.items()
    }

if hasattr(policy_from_script, "_queues"):
    policy_from_script._queues.clear()

cfg = PipelineConfig(
     D=8,
     N=10,
     Tf=1, Tbw=1, Tba=1, Tu=1,
     Tc=0, 
     stage_to_device=[0, 1, 2,3,3,2,1, 0],
 )

sim = PipelineSim(cfg)
script_gpu = build_dualpipev_script_gpu(D=cfg.D, N=cfg.N, stage_to_device=cfg.stage_to_device)
if hasattr(policy_from_script, "_queues"):
    policy_from_script._queues.clear()
policy = partial(policy_from_script, script_gpu=script_gpu)


for t in range(100):
    sim.step(t, policy)
    if sim.done():
        break
sim.print_summary()
sim.print_memory_timeline()

## Interconnecting our simulator with PyTorch

We propose to interconnect our simulator with **PyTorch Distributed Pipelining**.

Next, we will integrate the following code into **`torch.distributed.pipelining`**.  
First, make sure you can run the referenced implementation, and then try one pipelining strategy on CPU:

- https://github.com/pytorch/pytorch/blob/main/torch/distributed/pipelining/schedules.py#L33-L39

**Question 10 — Extract the CSV of a Pipelining strategy from PyTorch**

Using the code below:
1. **Extract `ScheduleDualPipeV` as a CSV** (dump the schedule to a `.csv` file).
2. **Open the CSV** and **compare it** against the schedule produced by our simulator.


In [ ]:
# run with  torchrun --standalone --nproc-per-node=4 test_schedule.py
import torch
import torch.nn as nn
import torch.distributed as dist

from torch.distributed.pipelining import PipelineStage

# ScheduleDualPipeV import location may vary by torch version
try:
    from torch.distributed.pipelining import ScheduleDualPipeV
except Exception:
    from torch.distributed.pipelining.schedules import ScheduleDualPipeV  # type: ignore


def init_dist():
    dist.init_process_group("gloo")
    rank = dist.get_rank()
    world = dist.get_world_size()
    device = torch.device("cpu")
    return rank, world, device


class M8(nn.Module):
    """8 sequential stages: 7x (Linear+GELU) + 1x Linear."""
    def __init__(self, d=16):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(7):
            self.blocks.append(nn.Sequential(nn.Linear(d, d), nn.GELU()))
        self.blocks.append(nn.Sequential(nn.Linear(d, d)))  # last stage

    def forward(self, x):
        for b in self.blocks:
            x = b(x)
        return x


def stage_module(full: M8, stage_idx: int) -> nn.Module:
    return full.blocks[stage_idx]


def loss_fn(pred, target):
    return torch.mean((pred - target) ** 2)


def main():
    rank, world, device = init_dist()
    assert world == 4, f"Run with --nproc-per-node=4. Got world_size={world}."

    # DualPipeV uses 2 stages per rank => D = 2*world = 8
    D = 2 * world  # 8
    full = M8().to(device)

    # V-mapping: rank r owns stages (r, D-1-r)
    s0, s1 = rank, (D - 1 - rank)

    stages = []
    for s in (s0, s1):
        mod = stage_module(full, s).to(device)
        stages.append(PipelineStage(mod, s, D, device, group=dist.group.WORLD))

    n_microbatches = 8  # must satisfy n_microbatches >= D for DualPipeV
    sched = ScheduleDualPipeV(stages, n_microbatches, loss_fn=loss_fn)

    # Dump compute-only schedule CSV from rank 0
    if rank == 0:
        out_csv = f"dualpipev_order_D{D}_world{world}_gloo.csv"
        if hasattr(sched, "_dump_csv"):
            try:
                sched._dump_csv(out_csv, format="compute_only")  # type: ignore[attr-defined]
            except TypeError:
                sched._dump_csv(out_csv)  # type: ignore[attr-defined]
        else:
            raise RuntimeError("This torch version does not expose _dump_csv on the schedule.")
        print("Wrote:", out_csv)

    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main()


**Question 11 — Import a Torch pipeline schedule into our discrete-time simulator.**  

You are given a CSV exported from PyTorch’s pipeline scheduler (one row per rank/GPU).  
Each cell is an action token such as `0F0`, `7I2`, `5W3`, `3B4`, or an overlapped token like  
`(0F7;7B3)OVERLAP_F_B`.

1. Write a converter that reads this CSV and produces a `script_gpu` dictionary compatible with our
   `policy_from_script` interface, i.e. `{gpu_id: "stageOp stageOp ... "}` with `Op ∈ {F, A, W, U}`.
2. Define the translation rules:
   - `F` → `F`
   - `I` (backward-input) → `A`
   - `W` → `W`
   - `B` (full backward) → expand into `A` then `W`
   - `OVERLAP_F_B` → expand its sub-actions in order (sequentialize the overlap)
   - microbatch indices can be dropped (our scripted policy ignores them)
3. Plug the resulting `script_gpu` into the simulator using  
   `policy = partial(policy_from_script, script_gpu=script_gpu)`  
   and run the simulation for `world_size=4` (i.e., `D=8` stages with V-mapping: rank `r` owns stages `(r, D−1−r)`).
4. Ensure your generated script appends the required `U` operations at the end.


In [ ]:
import re, csv
from collections import defaultdict
from functools import partial

# -----------------------------
# CSV (Torch schedule) -> script_gpu (our simulator)
# -----------------------------
def torch_csv_to_script_gpu(csv_path: str, add_updates: bool = True):
    """
    Reads a Torch pipelining schedule CSV (one row per rank) and converts it to our
    policy_from_script format: {gpu/rank: "stageOp stageOp ..."} with Op in {F,A,W,U}.

    Mapping:
      F  -> F
      I  -> A            (BACKWARD_INPUT becomes our activation-backward)
      W  -> W
      B  -> A then W     (FULL_BACKWARD expands)
      OVERLAP_F_B -> sequentialize subactions (expand each subaction similarly)

    Note: our policy_from_script ignores microbatch ids, so we drop trailing mb indices.
    """
    def expand_one(tok: str):
        tok = tok.strip()
        if not tok:
            return []

        # Overlap: "(0F7;7B3)OVERLAP_F_B"
        if "OVERLAP_F_B" in tok:
            m = re.match(r"\((.*)\)OVERLAP_F_B", tok)
            if not m:
                raise ValueError(f"Bad OVERLAP token: {tok}")
            inside = m.group(1)
            parts = [p.strip() for p in inside.split(";") if p.strip()]
            out = []
            for p in parts:
                out.extend(expand_one(p))
            return out

        # Regular: "7I0", "3B5", "0F7", ...
        m = re.match(r"(\d+)([FIWB])(\d+)?$", tok)
        if not m:
            raise ValueError(f"Unrecognized token: {tok}")
        stage = int(m.group(1))
        typ = m.group(2)

        if typ == "F":
            return [(stage, "F")]
        if typ == "I":
            return [(stage, "A")]
        if typ == "W":
            return [(stage, "W")]
        if typ == "B":
            return [(stage, "A"), (stage, "W")]
        raise ValueError(f"Unknown type: {typ}")

    # Read CSV
    with open(csv_path, newline="") as f:
        rows = list(csv.reader(f))

    # Fill in here
    
    return script_gpu


# -----------------------------
# Example: plug into our simulator
# -----------------------------
# 1) Convert your CSV to script_gpu
csv_path = "dualpipev_order_D8_world4_gloo.csv"  # <-- change to your filename
script_gpu = torch_csv_to_script_gpu(csv_path)
print("script_gpu[0] head:", " ".join(script_gpu[0].split()[:20]), "...")

# 2) Configure simulator for world_size=4 DualPipeV (D=8 stages, 2 per GPU, V mapping)
D, N = 8, 8
cfg = PipelineConfig(
    D=D, N=N,
    Tf=1, Tbw=1, Tba=1, Tu=1,
    Tc=0,
    stage_to_device=[0, 1, 2, 3, 3, 2, 1, 0],  # V mapping: rank r owns (r, D-1-r)
    serialize_send_per_gpu=True,
)

sim = PipelineSim(cfg)
policy = partial(policy_from_script, script_gpu=script_gpu)

for t in range(500):
    sim.step(t, policy)
    if sim.done():
        break

sim.print_summary(show_time_header=True)
sim.print_memory_timeline()


We could also write a script to convert our GPU script as follows:

In [ ]:
import csv
import re
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

def script_gpu_to_torch_csv(
    script_gpu: Dict[int, str],
    out_csv_path: str = "script_as_torch_compute_only.csv",
    ranks: Optional[List[int]] = None,
) -> str:
    """
    Convert our script_gpu (policy_from_script format) into a Torch Distributed
    'compute_only' CSV (one row per rank).

    Input tokens: "<stage><op>" with op in {F,A,W,U}
    Output tokens: "<stage><TorchOp><mb>" where TorchOp in {F,I,W,B}
      - A -> I
      - F -> F
      - W -> W
      - U is dropped by default (Torch schedules typically don't encode optimizer steps)

    Note: since the script does not include microbatch ids, we assign mb ids
    by per-stage counters (the k-th time a stage runs F becomes Fk, etc.).
    """
    # Choose which ranks (rows) to write
    if ranks is None:
        ranks = sorted(int(r) for r in script_gpu.keys())

    # Per-rank list of parsed (stage, op)
    parsed: Dict[int, List[Tuple[int, str]]] = {}
    for r in ranks:
        s = str(script_gpu.get(r, ""))
        pairs = re.findall(r"(\d+)\s*([FWAU])", s)
        parsed[r] = [(int(st), op) for st, op in pairs]

    # Per-rank per-stage counters for mb ids
    # Torch expects integer microbatch indices.
    f_cnt = defaultdict(int)
    i_cnt = defaultdict(int)
    w_cnt = defaultdict(int)

    # Build rows
    rows: List[List[str]] = []
    max_len = 0

    for r in ranks:
        row: List[str] = []
        for stage, op in parsed[r]:
            if op == "U":
                continue

            if op == "F":
                mb = f_cnt[(r, stage)]
                row.append(f"{stage}F{mb}")
                f_cnt[(r, stage)] += 1

            elif op == "A":
                mb = i_cnt[(r, stage)]
                row.append(f"{stage}I{mb}")   # Torch uses I for backward-input
                i_cnt[(r, stage)] += 1

            elif op == "W":
                mb = w_cnt[(r, stage)]
                row.append(f"{stage}W{mb}")
                w_cnt[(r, stage)] += 1

            else:
                # ignore anything unexpected
                continue

        rows.append(row)
        max_len = max(max_len, len(row))

    # Make it a proper rectangular CSV (Torch CSV loaders usually expect same length rows)
    for row in rows:
        if len(row) < max_len:
            row.extend([""] * (max_len - len(row)))

    with open(out_csv_path, "w", newline="") as f:
        w = csv.writer(f)
        for row in rows:
            w.writerow(row)

    return out_csv_path


cfg = PipelineConfig(
     D=8,
     N=10,
     Tf=1, Tbw=1, Tba=1, Tu=1,
     Tc=0, 
     stage_to_device=[0, 1, 2,3,3,2,1, 0],
 )

sim = PipelineSim(cfg)
script_gpu = build_dualpipev_script_gpu(D=cfg.D, N=cfg.N, stage_to_device=cfg.stage_to_device)
script_gpu_to_torch_csv(script_gpu, out_csv_path = "mycsv.csv", ranks=[0, 1, 2,3,3,2,1, 0])